# Modelo baseline - Dummy Classifier

In [69]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import cross_validate  
import numpy as np 
from sklearn.model_selection import StratifiedKFold

In [70]:
import pandas as pd

path = '../data/raw/Telco_customer_churn.xlsx'

df = pd.read_excel(path)
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [71]:
df.rename(columns={
            'CustomerID': 'customer_id',
            'Count': 'count',	
            'Country': 'country',	
            'State': 'state',
            'City': 'city',
            'Zip Code': 'zip_code',
            'Lat Long': 'lat_long',
            'Latitude': 'latitude',
            'Longitude': 'longitude',	
            'Gender': 'gender',
            'Senior Citizen': 'senior_citizen',
            'Partner': 'partner',	
            'Dependents': 'dependents',	
            'Tenure Months': 'tenure_months',	
            'Phone Service': 'phone_service',	
            'Multiple Lines': 'multiple_lines',	
            'Internet Service': 'internet_service',	
            'Online Security': 'online_security',	
            'Online Backup': 'online_backup',	
            'Device Protection': 'device_protection',	
            'Tech Support': 'tech_support',	
            'Streaming TV': 'streaming_tv',	
            'Streaming Movies': 'streaming_movies',	
            'Contract': 'contract',	
            'Paperless Billing': 'paperless_billing',	
            'Payment Method': 'payment_method',
            'Monthly Charges': 'monthly_charges',	
            'Total Charges': 'total_charges',	
            'Churn Label': 'churn_label',	
            'Churn Value': 'churn_value',	
            'Churn Score': 'churn_score',	
            'CLTV': 'cltv',	
            'Churn Reason': 'churn_reason'},
            inplace=True
)

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   object 
 1   count              7043 non-null   int64  
 2   country            7043 non-null   object 
 3   state              7043 non-null   object 
 4   city               7043 non-null   object 
 5   zip_code           7043 non-null   int64  
 6   lat_long           7043 non-null   object 
 7   latitude           7043 non-null   float64
 8   longitude          7043 non-null   float64
 9   gender             7043 non-null   object 
 10  senior_citizen     7043 non-null   object 
 11  partner            7043 non-null   object 
 12  dependents         7043 non-null   object 
 13  tenure_months      7043 non-null   int64  
 14  phone_service      7043 non-null   object 
 15  multiple_lines     7043 non-null   object 
 16  internet_service   7043 

In [73]:
df['monthly_charges'] = df['monthly_charges'].astype('float64')
df['latitude'] = df['latitude'].astype('float64')
df['longitude'] = df['longitude'].astype('float64')
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

In [74]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Remover colunas problemáticas
df_model = df.drop(columns=[
    'customer_id',
    'lat_long',
    'latitude',
    'longitude',
    'zip_code',
    'churn_reason',
    'churn_score'
])

# Variável alvo
y = df_model['churn_value']

# Features
X = df_model.drop(columns=['churn_value'])

# Identificar colunas
categorical_cols = X.select_dtypes(include='object').columns
numeric_cols = X.select_dtypes(exclude='object').columns

# Garantir que categóricas sejam string
X[categorical_cols] = X[categorical_cols].astype(str)

# Criar transformador
preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# Aplicar transformação
x = preprocessador.fit_transform(X)

In [75]:
x = preprocessador.fit_transform(X)

x = pd.DataFrame(
    x.toarray(),
    columns=preprocessador.get_feature_names_out()
)

x.head()

,cat__city_Acton,cat__city_Adelanto,cat__city_Adin,cat__city_Agoura Hills,cat__city_Aguanga,cat__city_Ahwahnee,cat__city_Alameda,cat__city_Alamo,cat__city_Albany,cat__city_Albion,...,cat__paperless_billing_Yes,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__churn_label_Yes,num__count,num__tenure_months,num__monthly_charges,num__total_charges,num__cltv
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,1.0,2.0,53.85,108.15,3239.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,2.0,70.70,151.65,2701.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,8.0,99.65,820.50,5372.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,28.0,104.80,3046.05,5003.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,49.0,103.70,5036.30,5340.0


In [76]:
# Verificação do nível de balanceamento da base de dados
print(f"Balanceamento de classes: {df['churn_value'].value_counts()}")

Balanceamento de classes: churn_value
0    5174
1    1869
Name: count, dtype: int64


In [77]:
# Definindo a semente para reprodutibilidade
SEED=20
np.random.seed(SEED)

# Criando o modelo Dummy Classifier
modelo_dummy = DummyClassifier(strategy='stratified', random_state=SEED)

# Definindo o método de validação cruzada
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# Realizando a validação cruzada
resultados = cross_validate(
    modelo_dummy,
    x,
    y,
    cv=cv,
    scoring=['accuracy','precision','recall','f1']
)

# Exibindo os resultados
print("Accuracy média:", resultados['test_accuracy'].mean())
print("F1 média:", resultados['test_f1'].mean())
print("Precision média:", resultados['test_precision'].mean())
print("Recall média:", resultados['test_recall'].mean())

Accuracy média: 0.608263418762089
F1 média: 0.28132343342036553
Precision média: 0.2741116751269036
Recall média: 0.2889253061928584
